# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore available record sets (by @id), fields and columns
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata. Attempting to read from the file objects...")
    # Use the Dataset's data_files (Croissant v1.0 compatibility)
    data_files = getattr(metadata, "data_files", None)
    if data_files:
        for file_obj in data_files:
            print(f"FileObject @{file_obj['@id']}")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} | name: {rs.get('name')}")
        # List fields/columns in each record set if available
        fields = rs.get('fields') or rs.get('columns')
        if fields:
            for f in fields:
                print(f"  - Field/Column @id: {f['@id']} | name: {f.get('name')}")
        else:
            print("  No fields/columns found for this record set.")

# For this dataset, the recordSets might not be populated in the metadata recordSet field; following Croissant v1.0, look in the .record_sets or .data_files
# Let's infer record sets from dataset.record_sets property:
print("\nrecord_sets property on metadata:", metadata.record_sets)
# Print out all root-level metadata keys for orientation
print("\nAvailable metadata properties:", dir(metadata))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll attempt to get all available record set @id's.
# In this dataset, the recordSet field is likely empty, so let's use the dataset.records() generator to get the available IDs.

record_sets = dataset.record_sets
if not record_sets:
    # Sometimes the DataFrame is a single table, referenced as the main body.
    # Let's query the record_set_ids property of dataset (mlcroissant >=0.3.5 supports .record_sets).
    try:
        # In case dataset.record_sets is a dict
        record_set_ids = list(dataset._dataset.get('recordSet', []))
        if not record_set_ids:
            # Try to grab via file objects in .distribution
            distributions = dataset._dataset.get('distribution', [])
            print("Distribution objects available (likely referencing the main table):")
            for dist in distributions:
                print(f" - distribution @id: {dist['@id']}")
        else:
            print(f"Available recordSets by @id: {record_set_ids}")
    except Exception as exc:
        print("Error determining recordSet IDs:", exc)

# However, dataset.records() works even if no explicit record sets are exposed: let's list all DataFrames under the available record sets.
dataframes = {}
try:
    available_record_sets = dataset.list_record_sets()
    print("Available record set @id's:", available_record_sets)
except Exception as exc:
    print("Could not use dataset.list_record_sets():", exc)
    # Try default record set name
    available_record_sets = [None]  # fallback

for record_set_id in available_record_sets:
    try:
        print(f"\nLoading records for record set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame with {len(dataframes[record_set_id])} rows and {len(dataframes[record_set_id].columns)} columns.")
            print("Columns:", dataframes[record_set_id].columns.tolist())
            display_cols = dataframes[record_set_id].columns[:5].tolist()  # Print first 5 columns
            print(dataframes[record_set_id][display_cols].head())
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as exc:
        print(f"Error loading records for record set {record_set_id}: {exc}")

# For further analysis, pick the first available non-empty DataFrame
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break
if main_record_set_id is not None:
    print(f"\nMain record set for analysis: {main_record_set_id}\nColumns: {dataframes[main_record_set_id].columns.tolist()}")
else:
    print("No valid dataframes available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA: Filter, normalize, group
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id].copy()
    # Find a numeric field for analysis by checking column dtypes or names
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates:
        # Try to guess by column names
        for c in df.columns:
            if any(x in c.lower() for x in ["age", "interval", "count", "number", "years", "size"]):
                try:
                    df[c] = pd.to_numeric(df[c], errors='coerce')
                    if pd.api.types.is_numeric_dtype(df[c]):
                        numeric_candidates.append(c)
                except:
                    continue
    print("Numeric fields detected:", numeric_candidates)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        # Try filtering for values greater than a threshold
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())

        # Normalize the numeric column
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Pick a group-by field (e.g. 'Sex', 'tumor_location', or similar categorical field)
        group_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and c != numeric_field]
        print("Group-by candidates:", group_candidates)
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped filtered data by '{group_field}', showing mean {numeric_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA. Please check the data above.")
else:
    print("No main record set DataFrame available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if we have a main record set and a numeric field
if main_record_set_id is not None and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_candidates:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Not enough data to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to use the `mlcroissant` library to load and explore the FAIR² dataset on second primary colorectal cancer (CRC) in cancer survivors.
- We loaded the dataset via its Croissant schema URL, discovered available record sets and fields (by their `@id`), and loaded the primary tabular data into a DataFrame.
- We performed simple exploratory analysis including filtering, normalization, grouping, and basic visualization of a numeric clinical variable (e.g., age or interval).
- Such FAIR datasets described by Croissant schemas enable automatic programmatic access and standardized data science analyses for clinical research and beyond.